In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision.datasets import CIFAR10
from torchvision import transforms
import torch.utils.data as data
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import numpy as np
import os


In [ ]:
data_path = "./data_path"
checkpoint_path = "./checkpoint_path"

os.makedirs(checkpoint_path, exist_ok = True)
def make_seed(seed):
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

make_seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available()else torch.device("cpu")

print("using this device", device)



using this device cuda:0


In [ ]:
train_dataset = CIFAR10(root = data_path, download = True)
data_std = (train_dataset.data/255.0).std(axis = (0,1,2))
data_mean = (train_dataset.data/255.0).mean(axis = (0,1,2))


100%|██████████| 170M/170M [36:32<00:00, 77.8kB/s]


In [ ]:
train_transform = transforms.Compose([transforms.ToTensor(),
                                      transforms.RandomHorizontalFlip(p = 0.5),
                                      transforms.RandomResizedCrop((32,32), scale = (0.8, 1.0), ratio =(0.8, 1.0)),
                                      transforms.Normalize(data_mean, data_std)])

test_transform = transforms.Compose([transforms.ToTensor(),
                                     transforms.Normalize(data_mean, data_std)])


train_dataset = CIFAR10(root = data_path, train = True, download = True, transform = train_transform)
val_dataset = CIFAR10(root = data_path, train = True, download = True, transform = test_transform)
test_dataset = CIFAR10(root = data_path, train = False, download = True, transform = test_transform)


train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_set, _ = torch.utils.data.random_split(train_dataset, [train_size, val_size])
_, val_set = torch.utils.data.random_split(val_dataset, [train_size, val_size])


train_dataloader = data.DataLoader(train_set, batch_size = 256, shuffle = True, num_workers = 0, pin_memory_device = True)
val_dataloader = data.DataLoader(val_set, batch_size = 256, shuffle = False, num_workers = 0, pin_memory_device = True)
test_dataloader = data.DataLoader(test_dataset, batch_size = 256, shuffle = False, num_workers = 0, pin_memory_device = True)

print(train_dataloader)
print(val_dataloader)
print(test_dataloader)

In [ ]:
class ResNetBlock(nn.Module):
    def __init__(self, c_in, c_out, subsample=False):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, kernel_size=3, stride=(2 if subsample else 1), padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(c_out, c_out, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)

        if subsample or c_in != c_out:
            self.downsample = nn.Sequential(
                nn.Conv2d(c_in, c_out, kernel_size=1, stride=(2 if subsample else 1), bias=False),
                nn.BatchNorm2d(c_out)
            )
        else:
            self.downsample = nn.Identity()

    def forward(self, x):
        identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, num_classes=10, num_blocks_per_layer=[2, 2, 2], c_hidden=[64, 128, 256]):
        super().__init__()
        self.initial_conv = nn.Sequential(
            nn.Conv2d(3, c_hidden[0], kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(c_hidden[0]),
            nn.ReLU(inplace=True)
        )

        self.layers = []
        c_in = c_hidden[0]
        for i, num_blocks in enumerate(num_blocks_per_layer):
            c_out = c_hidden[i]
            for j in range(num_blocks):
                subsample = (j == 0 and i > 0)
                self.layers.append(ResNetBlock(c_in, c_out, subsample))
                c_in = c_out
        self.layers = nn.Sequential(*self.layers)

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(c_hidden[-1], num_classes)

    def forward(self, x):
        x = self.initial_conv(x)
        x = self.layers(x)
        x = self.avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

The `ResNet` class has been updated to a more robust architecture. It now includes `ResNetBlock` which defines a standard residual block with skip connections, batch normalization, and ReLU activations. The `ResNet` class itself now stacks multiple `ResNetBlock` instances to create a deeper network with increasing channel dimensions, which is typical for ResNet models designed for image classification tasks like CIFAR-10.

In [ ]:
model = ResNet(num_classes=10, num_blocks_per_layer=[2, 2, 2], c_hidden=[64, 128, 256]).to(device)
print(model)

ResNet(
  (initial_conv): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layers): Sequential(
    (0): ResNetBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (downsample): Identity()
    )
    (1): ResNetBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 

In [ ]:
model = ResNet(num_classes=10, num_blocks_per_layer=[2, 2, 2], c_hidden=[64, 128, 256]).to(device)
print(model)

ResNet(
  (initial_conv): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layers): Sequential(
    (0): ResNetBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (downsample): Identity()
    )
    (1): ResNetBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 

In [ ]:
model = ResNet().to(device)
model

ResNet(
  (initial_conv): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layers): Sequential(
    (0): ResNetBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (downsample): Identity()
    )
    (1): ResNetBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-2, weight_decay = 1e-5, momentum = 0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = 6, last_epoch = -1)
criterion = nn.CrossEntropyLoss()
num_epochs = 150

In [ ]:
import tqdm.notebook as tqdm

In [ ]:
print(model)

ResNet(
  (initial_conv): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layers): Sequential(
    (0): ResNetBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (downsample): Identity()
    )
    (1): ResNetBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 

In [ ]:
for epoch in tqdm.tqdm(range(num_epochs)):
  for data_input, data_label in train_dataloader:
    data_input = data_input.to(device)
    data_label = data_label.to(device)


    pred = model(data_input)
    loss_module = criterion(pred, data_label)
    optimizer.zero_grad()
    loss_module.backward()
    optimizer.step()
  scheduler.step()

  0%|          | 0/150 [00:00<?, ?it/s]

In [ ]:
with torch.no_grad():
  correct = 0
  total = 0
  for data_input, data_label in val_dataloader:
    data_input = data_input.to(device)
    data_label = data_label.to(device)
    pred = model(data_input)
    _, predicted = torch.max(pred.data, 1)
    total += (data_label.size(0))
    correct+= (predicted == data_label).sum().item()

  acc = 100 * correct / total

  print(f"the accuracy of the model is defined as {acc:.3f}%")

the accuracy of the model is defined as 92.770%
